### Programación para Ciencia de Datos

### Estructura de Datos

### Parte 1: Carga y Exploración de Datos 25

### Parte 2: Consultas y Filtros 25

### Parte 3: Cálculos y Estadísticas 25

### Parte 4: Identificación de Riesgo y Reportes 25

In [1]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from datetime import datetime
import json
import os

In [2]:
def cargar_datos() -> tuple:
    """
    Carga los datos de estudiantes, calificaciones y materias.
    Retorna: (df_estudiantes, df_calificaciones, df_materias)
    """
    estudiantes = pd.DataFrame({
        'boleta': ['2021630001','2021630002','2021630003','2021630004','2021630005',
                   '2022630001','2022630002','2022630003','2022630004','2022630005',
                   '2023630001','2023630002','2023630003','2023630004','2023630005'],
        'nombre': ['Juan Pérez García','María López Ruiz','Pedro Sánchez Torres',
                   'Ana Martínez Díaz','Luis Rodríguez Vega','Carmen Flores Luna',
                   'Roberto Díaz Mora','Laura Torres Silva','Diego Ramírez Cruz',
                   'Sofía Vargas Romo','Carlos Mendoza Ríos','Patricia Ortiz León',
                   'Miguel Ángel Castro','Fernanda Reyes Paz','Andrés Guzmán Villa'],
        'semestre': [4,4,4,4,4,3,3,3,3,3,2,2,2,2,2],
        'carrera': ['CD']*15,
        'email': ['juan.perez@ipn.mx','maria.lopez@ipn.mx','pedro.sanchez@ipn.mx',
                  'ana.martinez@ipn.mx','luis.rodriguez@ipn.mx','carmen.flores@ipn.mx',
                  'roberto.diaz@ipn.mx','laura.torres@ipn.mx','diego.ramirez@ipn.mx',
                  'sofia.vargas@ipn.mx','carlos.mendoza@ipn.mx','patricia.ortiz@ipn.mx',
                  'miguel.castro@ipn.mx','fernanda.reyes@ipn.mx','andres.guzman@ipn.mx']
    })

    materias = pd.DataFrame({
        'materia_id': ['MAT101','MAT102','PROG101','PROG102','EST101','EST102','BD101'],
        'nombre': ['Cálculo Diferencial','Cálculo Integral','Programación I',
                   'Programación II','Probabilidad','Estadística Inferencial','Bases de Datos'],
        'creditos': [8,8,6,6,6,6,6],
        'semestre_materia': [1,2,1,2,2,3,3]
    })

    np.random.seed(42)
    calificaciones_data = []
    for boleta in estudiantes['boleta']:
        semestre = estudiantes[estudiantes['boleta'] == boleta]['semestre'].values[0]
        materias_cursadas = materias[materias['semestre_materia'] <= semestre]['materia_id'].tolist()
        for materia in materias_cursadas:
            base = np.random.uniform(5, 10)
            p1 = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            p2 = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            final = round(min(10, max(0, base + np.random.normal(0, 0.5))), 1)
            if np.random.random() < 0.05:
                p2 = np.nan
            calificaciones_data.append({'boleta': boleta, 'materia_id': materia,
                                        'parcial_1': p1, 'parcial_2': p2, 'final': final})
    calificaciones = pd.DataFrame(calificaciones_data)
    return estudiantes, calificaciones, materias

In [3]:
def info_general(df_estudiantes: pd.DataFrame, df_calificaciones: pd.DataFrame) -> dict:
    """
    Genera información general del sistema.
    Retorna: total_estudiantes, total_registros_calif, semestres, materias_con_registros
    """
    return {
        "total_estudiantes": len(df_estudiantes),
        "total_registros_calif": len(df_calificaciones),
        "semestres": sorted(df_estudiantes['semestre'].unique().tolist()),
        "materias_con_registros": df_calificaciones['materia_id'].nunique()
    }

In [4]:
def validar_datos(df_calificaciones: pd.DataFrame) -> dict:
    """
    Valida la integridad de los datos.
    Detecta nulos y calificaciones fuera de rango (< 0 o > 10).
    """
    cols_calif = ['parcial_1', 'parcial_2', 'final']
    registros_con_nulos = int(df_calificaciones[cols_calif].isna().any(axis=1).sum())

    mask_fuera = pd.DataFrame(False, index=df_calificaciones.index, columns=cols_calif)
    for col in cols_calif:
        mask_fuera[col] = (df_calificaciones[col] < 0) | (df_calificaciones[col] > 10)
    calificaciones_fuera_rango = int(mask_fuera.any(axis=1).sum())

    datos_validos = (registros_con_nulos == 0) and (calificaciones_fuera_rango == 0)
    return {
        "registros_con_nulos": registros_con_nulos,
        "calificaciones_fuera_rango": calificaciones_fuera_rango,
        "datos_validos": datos_validos
    }

In [5]:
def buscar_estudiante(df_estudiantes: pd.DataFrame, criterio: str, valor: str) -> pd.DataFrame:
    """
    Busca estudiantes por criterio: 'boleta' (exacto), 'nombre' (parcial), 'semestre' (int).
    """
    if criterio == 'boleta':
        return df_estudiantes[df_estudiantes['boleta'] == valor].copy()
    elif criterio == 'nombre':
        return df_estudiantes[df_estudiantes['nombre'].str.contains(valor, case=False, na=False)].copy()
    elif criterio == 'semestre':
        return df_estudiantes[df_estudiantes['semestre'] == int(valor)].copy()
    else:
        return pd.DataFrame()

In [6]:
def _promedio_estudiante(df_calificaciones: pd.DataFrame) -> pd.Series:
    """Helper: promedio general por estudiante (promedio de promedios por materia)."""
    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)
    return df.groupby('boleta')['prom_materia'].mean()

def obtener_kardex(boleta: str, df_estudiantes: pd.DataFrame,
                   df_calificaciones: pd.DataFrame, df_materias: pd.DataFrame) -> dict:
    """
    Obtiene el kardex completo de un estudiante:
    datos personales, calificaciones por materia, promedios y créditos.
    """
    resultado = {
        "estudiante": None, "materias": None, "promedio_general": None,
        "creditos_cursados": None, "materias_aprobadas": None, "materias_reprobadas": None
    }
    fila_est = df_estudiantes[df_estudiantes['boleta'] == boleta]
    if fila_est.empty:
        return resultado

    resultado['estudiante'] = fila_est.iloc[0].to_dict()

    calif_est = df_calificaciones[df_calificaciones['boleta'] == boleta].copy()
    calif_est['prom_materia'] = calif_est[['parcial_1','parcial_2','final']].mean(axis=1)
    calif_est = calif_est.merge(df_materias[['materia_id','nombre','creditos']], on='materia_id', how='left')
    calif_est = calif_est.rename(columns={'nombre': 'materia'})

    resultado['materias'] = calif_est[['materia_id','materia','parcial_1','parcial_2','final','prom_materia','creditos']]
    resultado['promedio_general'] = round(float(calif_est['prom_materia'].mean()), 2)
    resultado['creditos_cursados'] = int(calif_est['creditos'].sum())
    resultado['materias_aprobadas'] = int((calif_est['prom_materia'] >= 6).sum())
    resultado['materias_reprobadas'] = int((calif_est['prom_materia'] < 6).sum())
    return resultado

In [7]:
def filtrar_por_rendimiento(df_calificaciones: pd.DataFrame,
                             df_estudiantes: pd.DataFrame,
                             min_promedio: float = None,
                             max_promedio: float = None) -> pd.DataFrame:
    """
    Filtra estudiantes cuyo promedio general está en [min_promedio, max_promedio].
    """
    promedios = _promedio_estudiante(df_calificaciones).reset_index()
    promedios.columns = ['boleta', 'promedio']
    if min_promedio is not None:
        promedios = promedios[promedios['promedio'] >= min_promedio]
    if max_promedio is not None:
        promedios = promedios[promedios['promedio'] <= max_promedio]
    return promedios.merge(df_estudiantes, on='boleta', how='left').sort_values('promedio', ascending=False)

In [8]:
def calcular_promedio_materia(df_calificaciones: pd.DataFrame, materia_id: str) -> dict:
    """
    Estadísticas completas de una materia: inscritos, promedios parciales,
    promedio general, tasa de aprobación, máximo y mínimo.
    """
    df = df_calificaciones[df_calificaciones['materia_id'] == materia_id].copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)
    return {
        "materia": materia_id,
        "inscritos": len(df),
        "promedio_parcial1": round(float(df['parcial_1'].mean()), 2),
        "promedio_parcial2": round(float(df['parcial_2'].mean()), 2),
        "promedio_final": round(float(df['final'].mean()), 2),
        "promedio_general": round(float(df['prom_materia'].mean()), 2),
        "tasa_aprobacion": round(float((df['prom_materia'] >= 6).mean() * 100), 1),
        "calificacion_maxima": round(float(df['prom_materia'].max()), 2),
        "calificacion_minima": round(float(df['prom_materia'].min()), 2)
    }

In [9]:
def ranking_estudiantes(df_calificaciones: pd.DataFrame,
                         df_estudiantes: pd.DataFrame,
                         top_n: int = 10) -> pd.DataFrame:
    """
    Top N estudiantes por promedio general, con posición, nombre y semestre.
    """
    promedios = _promedio_estudiante(df_calificaciones).reset_index()
    promedios.columns = ['boleta', 'promedio']
    promedios = promedios.sort_values('promedio', ascending=False).head(top_n).reset_index(drop=True)
    promedios['posicion'] = promedios.index + 1
    promedios['promedio'] = promedios['promedio'].round(2)
    return promedios.merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta', how='left')[
        ['posicion','nombre','semestre','promedio']]

In [10]:
def estadisticas_por_semestre(df_estudiantes: pd.DataFrame,
                               df_calificaciones: pd.DataFrame) -> pd.DataFrame:
    """
    Estadísticas agrupadas por semestre:
    número de estudiantes, promedio, tasa de aprobación, mejor y peor promedio.
    """
    promedios = _promedio_estudiante(df_calificaciones).reset_index()
    promedios.columns = ['boleta', 'promedio']

    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)
    tasa = df.groupby('boleta').apply(lambda x: (x['prom_materia'] >= 6).mean() * 100).reset_index()
    tasa.columns = ['boleta', 'tasa_aprobacion']

    merged = promedios.merge(tasa, on='boleta').merge(df_estudiantes[['boleta','semestre']], on='boleta')
    return merged.groupby('semestre').agg(
        estudiantes=('boleta','count'),
        promedio=('promedio', lambda x: round(x.mean(), 2)),
        tasa_aprobacion=('tasa_aprobacion', lambda x: round(x.mean(), 1)),
        mejor_promedio=('promedio', lambda x: round(x.max(), 2)),
        peor_promedio=('promedio', lambda x: round(x.min(), 2))
    )

In [11]:
def identificar_estudiantes_riesgo(df_calificaciones: pd.DataFrame,
                                    df_estudiantes: pd.DataFrame,
                                    umbral_promedio: float = 7.0,
                                    max_reprobadas: int = 2) -> pd.DataFrame:
    """
    Identifica estudiantes en riesgo:
    - Promedio < umbral_promedio
    - Más de max_reprobadas materias reprobadas (prom_materia < 6)
    Agrega columna 'motivo': Bajo promedio / Mat. reprob. / Ambos
    """
    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)
    promedios = df.groupby('boleta')['prom_materia'].mean().reset_index()
    promedios.columns = ['boleta','promedio']
    reprobadas = df[df['prom_materia'] < 6].groupby('boleta').size().reset_index()
    reprobadas.columns = ['boleta','reprobadas']

    risk = promedios.merge(reprobadas, on='boleta', how='left')
    risk['reprobadas'] = risk['reprobadas'].fillna(0).astype(int)

    bajo = risk['promedio'] < umbral_promedio
    muchas_rep = risk['reprobadas'] > max_reprobadas
    risk = risk[bajo | muchas_rep].copy()

    def motivo(row):
        b = row['promedio'] < umbral_promedio
        r = row['reprobadas'] > max_reprobadas
        if b and r: return 'Ambos'
        if b: return 'Bajo promedio'
        return 'Mat. reprob.'

    risk['motivo'] = risk.apply(motivo, axis=1)
    risk['promedio'] = risk['promedio'].round(2)
    return risk.merge(df_estudiantes[['boleta','nombre']], on='boleta', how='left')[
        ['boleta','nombre','promedio','reprobadas','motivo']]

In [12]:
def generar_reporte_academico(df_estudiantes: pd.DataFrame,
                               df_calificaciones: pd.DataFrame,
                               df_materias: pd.DataFrame) -> dict:
    """
    Reporte académico completo: resumen general, estadísticas por semestre y materia,
    top estudiantes y estudiantes en riesgo.
    """
    df = df_calificaciones.copy()
    df['prom_materia'] = df[['parcial_1','parcial_2','final']].mean(axis=1)

    reporte = {
        "resumen_general": {
            "total_estudiantes": len(df_estudiantes),
            "promedio_global": round(float(df['prom_materia'].mean()), 2),
            "tasa_aprobacion": round(float((df['prom_materia'] >= 6).mean() * 100), 1)
        },
        "por_semestre": estadisticas_por_semestre(df_estudiantes, df_calificaciones),
        "por_materia": pd.DataFrame([calcular_promedio_materia(df_calificaciones, m)
                                     for m in df_calificaciones['materia_id'].unique()]),
        "mejores_estudiantes": ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5),
        "estudiantes_riesgo": identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes),
        "fecha_generacion": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    return reporte

In [13]:
def exportar_kardex(boleta: str, kardex: dict, formato: str = 'csv') -> str:
    """
    Exporta el kardex a CSV o JSON.
    Retorna el nombre del archivo generado.
    """
    fecha = datetime.now().strftime("%Y%m%d_%H%M%S")
    nombre = f"kardex_{boleta}_{fecha}.{formato}"
    if kardex['materias'] is None:
        return "Error: kardex vacío"
    if formato == 'csv':
        kardex['materias'].to_csv(nombre, index=False)
    elif formato == 'json':
        data = {
            'estudiante': kardex['estudiante'],
            'materias': kardex['materias'].to_dict(orient='records'),
            'promedio_general': kardex['promedio_general'],
            'creditos_cursados': kardex['creditos_cursados'],
            'materias_aprobadas': kardex['materias_aprobadas'],
            'materias_reprobadas': kardex['materias_reprobadas']
        }
        with open(nombre, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    print(f"Archivo generado: {nombre}")
    return nombre

In [14]:
def mostrar_kardex(kardex: Dict) -> None:
    """Muestra el kardex de forma legible."""
    if kardex['estudiante'] is None:
        print("❌ Estudiante no encontrado")
        return
    
    est = kardex['estudiante']
    print("=" * 70)
    print("                         KARDEX ACADÉMICO")
    print("=" * 70)
    print(f"\n📋 DATOS DEL ESTUDIANTE")
    print("-" * 40)
    print(f"Boleta: {est.get('boleta', 'N/A')}")
    print(f"Nombre: {est.get('nombre', 'N/A')}")
    print(f"Semestre: {est.get('semestre', 'N/A')}")
    print(f"Carrera: {est.get('carrera', 'N/A')}")
    print(f"Email: {est.get('email', 'N/A')}")
    
    print(f"\n📚 CALIFICACIONES")
    print("-" * 70)
    if kardex['materias'] is not None and len(kardex['materias']) > 0:
        print(kardex['materias'].to_string(index=False))
    else:
        print("Sin calificaciones registradas")
    
    print(f"\n📊 RESUMEN")
    print("-" * 40)
    print(f"Promedio General: {kardex.get('promedio_general', 0):.2f}")
    print(f"Créditos Cursados: {kardex.get('creditos_cursados', 0)}")
    print(f"Materias Aprobadas: {kardex.get('materias_aprobadas', 0)}")
    print(f"Materias Reprobadas: {kardex.get('materias_reprobadas', 0)}")
    print("=" * 70)

In [15]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte académico completo."""
    print("=" * 70)
    print("              REPORTE ACADÉMICO - CIENCIA DE DATOS")
    print(f"              Generado: {reporte['fecha_generacion']}")
    print("=" * 70)
    
    res = reporte.get('resumen_general', {})
    print(f"\n📊 RESUMEN GENERAL")
    print("-" * 40)
    print(f"Total de estudiantes: {res.get('total_estudiantes', 'N/A')}")
    print(f"Promedio global: {res.get('promedio_global', 0):.2f}")
    print(f"Tasa de aprobación: {res.get('tasa_aprobacion', 0):.1f}%")
    
    if reporte.get('por_semestre') is not None:
        print(f"\n📅 ESTADÍSTICAS POR SEMESTRE")
        print("-" * 40)
        print(reporte['por_semestre'].to_string())
    
    if reporte.get('mejores_estudiantes') is not None:
        print(f"\n🏆 TOP 5 ESTUDIANTES")
        print("-" * 40)
        print(reporte['mejores_estudiantes'].head().to_string(index=False))
    
    if reporte.get('estudiantes_riesgo') is not None and len(reporte['estudiantes_riesgo']) > 0:
        print(f"\n⚠️ ESTUDIANTES EN RIESGO ({len(reporte['estudiantes_riesgo'])})")
        print("-" * 40)
        print(reporte['estudiantes_riesgo'].to_string(index=False))
    else:
        print(f"\n✅ No hay estudiantes en riesgo académico")
    
    print("\n" + "=" * 70)

In [16]:
df_estudiantes, df_calificaciones, df_materias = cargar_datos()

print("DATOS CARGADOS")
print("=" * 50)
print(f"\nEstudiantes ({len(df_estudiantes)} registros):")
print(df_estudiantes.head())

print(f"\nCalificaciones ({len(df_calificaciones)} registros):")
print(df_calificaciones.head())

print(f"\nMaterias ({len(df_materias)} registros):")
print(df_materias)

DATOS CARGADOS

Estudiantes (15 registros):
       boleta                nombre  semestre carrera                  email
0  2021630001     Juan Pérez García         4      CD      juan.perez@ipn.mx
1  2021630002      María López Ruiz         4      CD     maria.lopez@ipn.mx
2  2021630003  Pedro Sánchez Torres         4      CD   pedro.sanchez@ipn.mx
3  2021630004     Ana Martínez Díaz         4      CD    ana.martinez@ipn.mx
4  2021630005   Luis Rodríguez Vega         4      CD  luis.rodriguez@ipn.mx

Calificaciones (95 registros):
       boleta materia_id  parcial_1  parcial_2  final
0  2021630001     MAT101        5.8        7.2    7.0
1  2021630001     MAT102        6.1        4.5    4.8
2  2021630001    PROG101        3.9        7.5    6.9
3  2021630001    PROG102        4.9        5.8    5.4
4  2021630001     EST101        8.6        6.4    6.1

Materias (7 registros):
  materia_id                   nombre  creditos  semestre_materia
0     MAT101      Cálculo Diferencial         8

In [17]:
print("\nINFORMACIÓN GENERAL")
print("=" * 50)
info = info_general(df_estudiantes, df_calificaciones)
print(info)


INFORMACIÓN GENERAL
{'total_estudiantes': 15, 'total_registros_calif': 95, 'semestres': [2, 3, 4], 'materias_con_registros': 7}


In [18]:
print("\nVALIDACIÓN DE DATOS")
print("=" * 50)
validacion = validar_datos(df_calificaciones)
print(validacion)


VALIDACIÓN DE DATOS
{'registros_con_nulos': 5, 'calificaciones_fuera_rango': 0, 'datos_validos': False}


In [19]:
print("\nBÚSQUEDA DE ESTUDIANTES")
print("=" * 50)

print("\n-- Buscar por nombre 'María' --")
resultado = buscar_estudiante(df_estudiantes, 'nombre', 'María')
print(resultado)

print("\n-- Buscar por semestre 3 --")
resultado = buscar_estudiante(df_estudiantes, 'semestre', '3')
print(resultado)


BÚSQUEDA DE ESTUDIANTES

-- Buscar por nombre 'María' --
       boleta            nombre  semestre carrera               email
1  2021630002  María López Ruiz         4      CD  maria.lopez@ipn.mx

-- Buscar por semestre 3 --
       boleta              nombre  semestre carrera                 email
5  2022630001  Carmen Flores Luna         3      CD  carmen.flores@ipn.mx
6  2022630002   Roberto Díaz Mora         3      CD   roberto.diaz@ipn.mx
7  2022630003  Laura Torres Silva         3      CD   laura.torres@ipn.mx
8  2022630004  Diego Ramírez Cruz         3      CD  diego.ramirez@ipn.mx
9  2022630005   Sofía Vargas Romo         3      CD   sofia.vargas@ipn.mx


In [20]:
print("\nKARDEX DE ESTUDIANTE")
print("=" * 50)
kardex = obtener_kardex('2021630001', df_estudiantes, df_calificaciones, df_materias)
mostrar_kardex(kardex)

print("\nEXPORTACIÓN DE KARDEX")
print("=" * 50)
archivo_csv = exportar_kardex('2021630001', kardex, 'csv')
archivo_json = exportar_kardex('2021630001', kardex, 'json')


KARDEX DE ESTUDIANTE
                         KARDEX ACADÉMICO

📋 DATOS DEL ESTUDIANTE
----------------------------------------
Boleta: 2021630001
Nombre: Juan Pérez García
Semestre: 4
Carrera: CD
Email: juan.perez@ipn.mx

📚 CALIFICACIONES
----------------------------------------------------------------------
materia_id                 materia  parcial_1  parcial_2  final  prom_materia  creditos
    MAT101     Cálculo Diferencial        5.8        7.2    7.0      6.666667         8
    MAT102        Cálculo Integral        6.1        4.5    4.8      5.133333         8
   PROG101          Programación I        3.9        7.5    6.9      6.100000         6
   PROG102         Programación II        4.9        5.8    5.4      5.366667         6
    EST101            Probabilidad        8.6        6.4    6.1      7.033333         6
    EST102 Estadística Inferencial        4.8        4.7    5.8      5.100000         6
     BD101          Bases de Datos        7.4        8.3    8.2      7.9

In [21]:
print("\nRANKING DE ESTUDIANTES")
print("=" * 50)
ranking = ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5)
print(ranking)


RANKING DE ESTUDIANTES
   posicion               nombre  semestre  promedio
0         1  Miguel Ángel Castro         2      8.91
1         2   Fernanda Reyes Paz         2      8.59
2         3  Luis Rodríguez Vega         4      8.44
3         4  Andrés Guzmán Villa         2      8.43
4         5   Laura Torres Silva         3      7.96


In [22]:
print("\nREPORTE ACADÉMICO COMPLETO")
reporte = generar_reporte_academico(df_estudiantes, df_calificaciones, df_materias)
mostrar_reporte(reporte)


REPORTE ACADÉMICO COMPLETO
              REPORTE ACADÉMICO - CIENCIA DE DATOS
              Generado: 2026-06-23 03:25:27

📊 RESUMEN GENERAL
----------------------------------------
Total de estudiantes: 15
Promedio global: 7.64
Tasa de aprobación: 89.5%

📅 ESTADÍSTICAS POR SEMESTRE
----------------------------------------
          estudiantes  promedio  tasa_aprobacion  mejor_promedio  peor_promedio
semestre                                                                       
2                   5      7.96             92.0            8.91           6.46
3                   5      7.75             91.4            7.96           7.41
4                   5      7.30             85.7            8.44           6.20

🏆 TOP 5 ESTUDIANTES
----------------------------------------
 posicion              nombre  semestre  promedio
        1 Miguel Ángel Castro         2      8.91
        2  Fernanda Reyes Paz         2      8.59
        3 Luis Rodríguez Vega         4      8.44
        4 An

### Bonus: Funcionalidades Extra Opcional

In [23]:
def predecir_riesgo_proximo_semestre(df_calificaciones: pd.DataFrame,
                                      df_estudiantes: pd.DataFrame) -> pd.DataFrame:
    """
    Predice estudiantes en riesgo el próximo semestre basándose en tendencias:
    - Tendencia decreciente en parciales (p1 > p2 > final) en >= 2 materias
    - Final menor que parcial_1 en >= 3 materias
    """
    df = df_calificaciones.dropna(subset=['parcial_1','parcial_2','final']).copy()
    df['tendencia_decreciente'] = (df['parcial_1'] > df['parcial_2']) & (df['parcial_2'] > df['final'])
    df['final_menor_p1'] = df['final'] < df['parcial_1']
    risk_flags = df.groupby('boleta').agg(
        materias_tendencia=('tendencia_decreciente', 'sum'),
        materias_final_menor=('final_menor_p1', 'sum')
    ).reset_index()
    riesgo = risk_flags[(risk_flags['materias_tendencia'] >= 2) | (risk_flags['materias_final_menor'] >= 3)]
    return riesgo.merge(df_estudiantes[['boleta','nombre','semestre']], on='boleta', how='left')

print("BONUS 1: PREDICTOR DE RIESGO PRÓXIMO SEMESTRE")
print("=" * 50)
prediccion = predecir_riesgo_proximo_semestre(df_calificaciones, df_estudiantes)
print(prediccion.to_string(index=False))

BONUS 1: PREDICTOR DE RIESGO PRÓXIMO SEMESTRE
    boleta  materias_tendencia  materias_final_menor               nombre  semestre
2021630002                   0                     4     María López Ruiz         4
2021630003                   1                     4 Pedro Sánchez Torres         4
2021630005                   0                     3  Luis Rodríguez Vega         4
2022630001                   0                     4   Carmen Flores Luna         3
2022630002                   1                     4    Roberto Díaz Mora         3
2022630004                   2                     3   Diego Ramírez Cruz         3
2022630005                   0                     5    Sofía Vargas Romo         3
2023630001                   0                     4  Carlos Mendoza Ríos         2


In [24]:
def comparar_estudiantes(boleta1: str, boleta2: str,
                          df_calificaciones: pd.DataFrame,
                          df_estudiantes: pd.DataFrame,
                          df_materias: pd.DataFrame) -> dict:
    """
    Compara el rendimiento académico de dos estudiantes:
    promedios, materias aprobadas/reprobadas y créditos.
    """
    k1 = obtener_kardex(boleta1, df_estudiantes, df_calificaciones, df_materias)
    k2 = obtener_kardex(boleta2, df_estudiantes, df_calificaciones, df_materias)
    return {
        "estudiante_1": {
            "nombre": k1['estudiante']['nombre'] if k1['estudiante'] else None,
            "promedio": k1['promedio_general'],
            "aprobadas": k1['materias_aprobadas'],
            "reprobadas": k1['materias_reprobadas'],
            "creditos": k1['creditos_cursados']
        },
        "estudiante_2": {
            "nombre": k2['estudiante']['nombre'] if k2['estudiante'] else None,
            "promedio": k2['promedio_general'],
            "aprobadas": k2['materias_aprobadas'],
            "reprobadas": k2['materias_reprobadas'],
            "creditos": k2['creditos_cursados']
        },
        "mejor_promedio": (k1['estudiante']['nombre']
                           if (k1['promedio_general'] or 0) >= (k2['promedio_general'] or 0)
                           else k2['estudiante']['nombre'])
    }

print("BONUS 2: COMPARADOR DE ESTUDIANTES")
print("=" * 50)
comp = comparar_estudiantes('2021630001', '2021630002',
                             df_calificaciones, df_estudiantes, df_materias)
for k, v in comp.items():
    print(f"\n{k}:")
    if isinstance(v, dict):
        for kk, vv in v.items():
            print(f"  {kk}: {vv}")
    else:
        print(f"  {v}")

BONUS 2: COMPARADOR DE ESTUDIANTES

estudiante_1:
  nombre: Juan Pérez García
  promedio: 6.2
  aprobadas: 4
  reprobadas: 3
  creditos: 46

estudiante_2:
  nombre: María López Ruiz
  promedio: 6.86
  aprobadas: 7
  reprobadas: 0
  creditos: 46

mejor_promedio:
  María López Ruiz
